# 02 — Análisis Longitudinal

**Observatorio de Ciencia, Tecnología e Innovación — Grupo 7**

Este notebook analiza la evolución temporal de los investigadores reconocidos
por Minciencias a través de las **6 convocatorias (2013–2021)**.

**Contenido:**
1. Carga y preparación del dataset consolidado
2. Evolución del número total de investigadores por convocatoria
3. Tendencias por género, área y región
4. Análisis de movilidad en categoría de clasificación
5. Verificación de unicidad de identificadores (ID_PERSONA_PR)
6. Conclusiones longitudinales

In [ ]:
import sys
import pathlib

ROOT = pathlib.Path().resolve().parent  # raiz del repo (notebooks/../)
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ingesta import cargar_consolidado
from Transformacion import transformar

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (11, 5)

## 1. Carga de datos

In [ ]:
df_raw = cargar_consolidado()
df = transformar(df_raw)
print(f'Shape: {df.shape}')
df.head()

## 2. Evolución total de investigadores por convocatoria

In [ ]:
evolucion = (
    df.groupby('ANO_CONVO_INT')
    .size()
    .reset_index(name='Investigadores')
)
print(evolucion)

plt.figure(figsize=(8, 5))
plt.bar(
    evolucion['ANO_CONVO_INT'].astype(str),
    evolucion['Investigadores'],
    color='steelblue',
    edgecolor='white',
    width=0.5
)
for _, row in evolucion.iterrows():
    anio, val = row['ANO_CONVO_INT'], row['Investigadores']
    plt.text(str(anio), val + 100, f'{val:,}', ha='center', fontsize=11)
plt.title('Investigadores reconocidos por convocatoria')
plt.xlabel('Año de convocatoria')
plt.ylabel('Número de investigadores')
plt.tight_layout()
plt.show()

## 3. Tendencias por género, área y región

In [ ]:
# Evolución por género
genero_anio = (
    df.groupby(['ANO_CONVO_INT', 'NME_GENERO_PR'])
    .size()
    .reset_index(name='Cantidad')
)

plt.figure(figsize=(10, 5))
for genero, grupo in genero_anio.groupby('NME_GENERO_PR'):
    plt.plot(
        grupo['ANO_CONVO_INT'].astype(str),
        grupo['Cantidad'],
        marker='o',
        label=genero
    )
plt.title('Evolución de investigadores por género')
plt.xlabel('Año de convocatoria')
plt.ylabel('Número de investigadores')
plt.legend(title='Género')
plt.tight_layout()
plt.show()

In [ ]:
# Evolución por gran área (top 5)
top_areas = df['NME_GRAN_AREA_PR'].value_counts().head(5).index.tolist()

area_anio = (
    df[df['NME_GRAN_AREA_PR'].isin(top_areas)]
    .groupby(['ANO_CONVO_INT', 'NME_GRAN_AREA_PR'])
    .size()
    .reset_index(name='Cantidad')
)

plt.figure(figsize=(12, 5))
for area, grupo in area_anio.groupby('NME_GRAN_AREA_PR'):
    plt.plot(
        grupo['ANO_CONVO_INT'].astype(str),
        grupo['Cantidad'],
        marker='o',
        label=area
    )
plt.title('Evolución por gran área de conocimiento (top 5)')
plt.xlabel('Año de convocatoria')
plt.ylabel('Número de investigadores')
plt.legend(title='Gran área', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 4. Movilidad en categoría de clasificación

In [ ]:
clas_anio = (
    df.groupby(['ANO_CONVO_INT', 'NME_CLASIFICACION_PR'])
    .size()
    .unstack(fill_value=0)
)
print('Distribución de categorías por convocatoria:')
print(clas_anio)

clas_anio.T.plot(kind='bar', figsize=(12, 5), colormap='tab10')
plt.title('Categorías de clasificación por convocatoria')
plt.xlabel('Categoría')
plt.ylabel('Número de investigadores')
plt.xticks(rotation=35, ha='right')
plt.legend(title='Convocatoria')
plt.tight_layout()
plt.show()

## 5. Verificación de unicidad de ID_PERSONA_PR

In [ ]:
if 'ID_PERSONA_PR' in df.columns:
    # Investigadores que aparecen en más de una convocatoria
    presencia = (
        df.groupby('ID_PERSONA_PR')['ANO_CONVO_INT']
        .nunique()
        .rename('n_convocatorias')
    )
    print('Distribución de investigadores por número de convocatorias:')
    print(presencia.value_counts().sort_index())

    # Verificación de consistencia de edad
    # Para un investigador entre 2017 y 2019 la diferencia debería ser ~2 años
    df_multi = df[df['ID_PERSONA_PR'].isin(presencia[presencia > 1].index)]
    edad_delta = (
        df_multi.groupby('ID_PERSONA_PR')
        .apply(lambda g: g.sort_values('ANO_CONVO_INT')['EDAD_ANOS_PR'].diff().abs().max())
        .dropna()
    )
    print('\nEstadísticas de la diferencia de edad entre convocatorias:')
    print(edad_delta.describe())
    print(f'IDs con diferencia de edad > 6 años: {(edad_delta > 6).sum():,}')
else:
    print('Columna ID_PERSONA_PR no disponible en el dataset.')

## 6. Conclusiones longitudinales

In [ ]:
print('=== RESUMEN LONGITUDINAL ===')
for anio, g in df.groupby('ANO_CONVO_INT'):
    print(f'\nConvocatoria {int(anio)}:')
    print(f'  Registros       : {len(g):,}')
    if 'ID_PERSONA_PR' in g.columns:
        print(f'  IDs únicos      : {g["ID_PERSONA_PR"].nunique():,}')
    if 'NME_GENERO_PR' in g.columns:
        pct_fem = (g['NME_GENERO_PR'] == 'FEMENINO').mean() * 100
        print(f'  % Femenino      : {pct_fem:.1f}%')
    if 'EDAD_ANOS_PR' in g.columns:
        print(f'  Edad media      : {g["EDAD_ANOS_PR"].mean():.1f} años')

## 7. Panel longitudinal — Seguimiento individual por `ID_PERSONA_PR`

Tracking de investigadores entre las **6 convocatorias (2013–2021)**, cubriendo
los 5 periodos consecutivos: 2013→2014, 2014→2015, 2015→2017, 2017→2019, 2019→2021.

Para cada investigador presente en una convocatoria se registra qué ocurre en la siguiente:

| Resultado | Descripción |
|---|---|
| **Se mantiene** | Aparece en ambas con la misma categoría |
| **Sube** | Avanza a una categoría superior |
| **Baja** | Retrocede a una categoría inferior |
| **Desaparece** | No aparece en la convocatoria siguiente |

In [ ]:
import sys, pathlib as _pl
_ROOT = _pl.Path().resolve().parent
if str(_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(_ROOT / "src"))

from analisis.longitudinal import (
    construir_panel,
    comparar_periodo,
    resumen_tracking,
    tracking_por_categoria,
    tracking_todos_periodos,
    tasa_retencion_por_periodo,
)

panel = construir_panel(df)
print(f"Panel analítico: {len(panel):,} filas | {panel['ID_PERSONA_PR'].nunique():,} investigadores únicos")
print(f"Convocatorias en el panel: {sorted(panel['ANO_CONVO_INT'].unique())}")

# Investigadores nuevos por convocatoria
anios_panel = sorted(panel["ANO_CONVO_INT"].unique())
filas_ingreso = []
for i, anio in enumerate(anios_panel):
    ids_actual = set(panel.loc[panel["ANO_CONVO_INT"] == anio, "ID_PERSONA_PR"])
    ids_previos = set(panel.loc[panel["ANO_CONVO_INT"].isin(anios_panel[:i]), "ID_PERSONA_PR"])
    nuevos = len(ids_actual - ids_previos)
    filas_ingreso.append({"convocatoria": anio, "total": len(ids_actual), "nuevos": nuevos})

df_ingresos = pd.DataFrame(filas_ingreso)
df_ingresos["pct_nuevos"] = (df_ingresos["nuevos"] / df_ingresos["total"]).round(3)
print("\nInvestigadores por convocatoria:")
print(df_ingresos.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
x = df_ingresos["convocatoria"].astype(str)
ax.bar(x, df_ingresos["total"], label="Total", color="steelblue", alpha=0.7)
ax.bar(x, df_ingresos["nuevos"], label="Nuevos (sin historial previo)", color="tomato", alpha=0.85)
ax.set_title("Investigadores por convocatoria: total vs. nuevos ingresos")
ax.set_xlabel("Convocatoria")
ax.set_ylabel("Investigadores")
ax.legend()
plt.tight_layout()
plt.show()

### 7.1 Tracking entre periodos consecutivos

In [ ]:
resumen_all = tracking_todos_periodos(panel)
print(resumen_all.to_string(index=False))

# Gráfico comparativo de todos los periodos
colores_resultado = {
    "Se mantiene": "steelblue",
    "Desaparece": "gray",
    "Sube": "seagreen",
    "Baja": "tomato",
}
periodos = resumen_all["periodo"].unique()
fig, axes = plt.subplots(1, len(periodos), figsize=(18, 5), sharey=False)

for ax, periodo in zip(axes, periodos):
    sub = resumen_all[resumen_all["periodo"] == periodo].set_index("resultado")
    orden_plot = ["Se mantiene", "Desaparece", "Sube", "Baja"]
    cats = [r for r in orden_plot if r in sub.index]
    vals = [sub.loc[r, "n"] for r in cats]
    cols = [colores_resultado[r] for r in cats]
    ax.bar(cats, vals, color=cols)
    ax.set_title(periodo, fontsize=10)
    ax.set_ylabel("Investigadores")
    ax.tick_params(axis="x", rotation=30)

plt.suptitle("Tracking longitudinal — todos los periodos consecutivos", fontsize=13)
plt.tight_layout()
plt.show()

### 7.2 Tasas de retención por periodo

In [ ]:
retencion = tasa_retencion_por_periodo(panel)
print(retencion.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(
    retencion["periodo"],
    retencion["tasa_retencion"] * 100,
    marker="o",
    color="steelblue",
    linewidth=2,
    label="Retención (%)",
)
ax.bar(
    retencion["periodo"],
    retencion["nuevos_en_final"] / retencion["total_inicial"] * 100,
    alpha=0.4,
    color="tomato",
    label="Nuevos / total anterior (%)",
)
ax.set_ylim(0, 100)
ax.set_ylabel("Porcentaje (%)")
ax.set_title("Tasas de retención y nuevos ingresos entre convocatorias consecutivas")
ax.legend()
plt.tight_layout()
plt.show()

### 7.3 Desglose del tracking por categoría inicial

In [ ]:
anios_panel = sorted(panel["ANO_CONVO_INT"].unique())
for a0, a1 in zip(anios_panel, anios_panel[1:]):
    comp_tmp = comparar_periodo(panel, a0, a1)
    tabla_cat = tracking_por_categoria(comp_tmp)
    print(f"\nPeriodo {a0}–{a1}:")
    print(tabla_cat.to_string())

### 7.4 Exportar evidencias del panel longitudinal

In [ ]:
import pathlib

EVIDENCIAS = pathlib.Path("../../evidencias")
EVIDENCIAS.mkdir(exist_ok=True)

resumen_all.to_csv(EVIDENCIAS / "panel_longitudinal_resumen_todos_periodos.csv", index=False)
df_ingresos.to_csv(EVIDENCIAS / "panel_longitudinal_ingresos_por_convocatoria.csv", index=False)
retencion.to_csv(EVIDENCIAS / "panel_longitudinal_tasas_retencion.csv", index=False)

anios_panel = sorted(panel["ANO_CONVO_INT"].unique())
for a0, a1 in zip(anios_panel, anios_panel[1:]):
    comp_tmp = comparar_periodo(panel, a0, a1)
    tracking_por_categoria(comp_tmp).to_csv(
        EVIDENCIAS / f"panel_longitudinal_cat_{a0}_{a1}.csv"
    )

print("Evidencias exportadas a:", EVIDENCIAS.resolve())